We observe comb pilots every 4 subcarriers in a 64-tone OFDM symbol ($N=64$, $N_p=16$), and estimate all tones, especially the 48 non-pilot tones.

Given pilot-domain model
$$
\mathbf{y}_p=\mathbf{X}_p\mathbf{h}_p+\mathbf{n}_p,\quad \mathbf{n}_p\sim \mathcal{CN}(\mathbf{0},\sigma_n^2\mathbf{I}),
$$
with unit-modulus known pilots ($\mathbf{X}_p=\mathbf{I}$), we recover full-band $\hat{\mathbf{H}}\in\mathbb{C}^{64}$ and evaluate
$$
\mathrm{NMSE}_t=\frac{\|\hat{\mathbf{H}}_D^{(t)}-\mathbf{H}_D^{(t)}\|_2^2}{\|\mathbf{H}_D^{(t)}\|_2^2},
$$
on non-pilot index set $D$.

Simulation assumptions:
- SISO OFDM, $N=64$, $\Delta f=15$ kHz, bandwidth $0.96$ MHz, CP = 16
- 3GPP-like TDL-A taps (continuous delays), quasi-static per OFDM symbol
- Monte Carlo trials per SNR: 500
- SNR sweep: 0 to 30 dB in 2 dB steps
- Synthetic data only

Key formulas implemented:

- Pilot observation:
  $$
  \mathbf{y}_p=\mathbf{X}_p\mathbf{h}_p+\mathbf{n}_p
  $$

- LS on pilot tones:
  $$
  \hat{\mathbf{h}}_p^{\mathrm{LS}}=\mathbf{X}_p^{-1}\mathbf{y}_p=\mathbf{y}_p \quad (\text{unit pilots})
  $$

- Linear interpolation baseline:
  $$
  \hat H_{\mathrm{lin}}[k]=\mathcal{I}_{\mathrm{lin}}\{(k_{p,m},\hat H_p^{\mathrm{LS}}[m])\}(k)
  $$

- Spline interpolation baseline:
  $$
  \hat H_{\mathrm{spline}}[k]=\sum_{m=0}^{N_p-1}\hat H_p^{\mathrm{LS}}[m]\,
  \beta_3\!\left(\frac{k-k_{p,m}}{\Delta k}\right)
  $$

- DFT delay truncation:
  $$
  \tilde h[n]=\frac{1}{N}\sum_{k=0}^{N-1}\hat H_{\mathrm{lin}}[k]e^{j2\pi kn/N},
  \quad
  \hat h_T[n]=\tilde h[n]\mathbf{1}_{0\le n<L_t},
  \quad
  \hat H_{\mathrm{DFT}}[k]=\sum_{n=0}^{N-1}\hat h_T[n]e^{-j2\pi kn/N}
  $$

- LMMSE Wiener interpolation from PDP covariance:
  $$
  [\mathbf{R}_{HH}]_{m,n}=\sum_{\ell=0}^{L_h-1}p_\ell e^{-j2\pi(m-n)\ell/N},
  \quad
  \hat{\mathbf{H}}_{\mathrm{LMMSE}}
  =
  \mathbf{R}_{Hp}\left(\mathbf{R}_{pp}+\sigma_n^2\mathbf{I}\right)^{-1}
  \hat{\mathbf{h}}_p^{\mathrm{LS}}
  $$

- Averaging and confidence interval:
  $$
  \overline{\mathrm{NMSE}}=\frac{1}{T}\sum_{t=1}^T\mathrm{NMSE}_t,
  \quad
  \mathrm{NMSE}_{dB}=10\log_{10}(\overline{\mathrm{NMSE}}),
  \quad
  \mathrm{CI}_{95\%}=\bar x\pm1.96\frac{s}{\sqrt{T}}
  $$

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from scipy.interpolate import CubicSpline
from scipy.linalg import toeplitz

Next, we define data generation for one OFDM symbol under a given operating point (SNR, pilot count, delay spread).

**Inputs:** $N$, $\Delta f$, pilot indices, SNR, PDP/delays.  
**Outputs:** true CFR $\mathbf{H}$, noisy pilot observations $\mathbf{y}_p$, LS pilot estimate $\hat{\mathbf{h}}_p^{LS}$, noise variance $\sigma_n^2$.  
Model equations used:
- $H[k]=\sum_{\ell}\alpha_\ell e^{-j2\pi k\Delta f\tau_\ell}$
- $\mathbf{y}_p=\mathbf{X}_p\mathbf{h}_p+\mathbf{n}_p$
- $\mathbf{n}_p=(\sigma/\sqrt{2})(\mathbf{u}+j\mathbf{v})$

In [ ]:
def get_tdl_a_profile(rms_delay_spread_ns=30.0):
    """
    Returns continuous-delay PDP approximating 3GPP TDL-A shape, then scales delays
    to match requested RMS delay spread.
    """
    # Relative delays (ns) and powers (dB), TDL-A-like
    delays_ns_base = np.array([0, 30, 70, 90, 110, 190, 410], dtype=float)
    powers_db = np.array([0, -1, -2, -3, -8, -17.2, -20.8], dtype=float)

    p_lin = 10 ** (powers_db / 10.0)
    p_lin = p_lin / np.sum(p_lin)

    # Scale delays to hit target RMS spread approximately
    mean_tau = np.sum(delays_ns_base * p_lin)
    rms_base = np.sqrt(np.sum(((delays_ns_base - mean_tau) ** 2) * p_lin))
    scale = rms_delay_spread_ns / (rms_base + 1e-15)
    delays_ns = delays_ns_base * scale

    return delays_ns * 1e-9, p_lin  # seconds, normalized powers


def pilot_indices_uniform(N, num_pilots):
    step = N // num_pilots
    return np.arange(0, N, step, dtype=int)


def generate_ofdm_observation(
    snr_db,
    N=64,
    delta_f_hz=15e3,
    num_pilots=16,
    rms_delay_spread_ns=30.0
):
    """
    Generate one OFDM-symbol channel realization and pilot observations.
    """
    k_all = np.arange(N)
    k_p = pilot_indices_uniform(N, num_pilots)

    delays_s, p_lin = get_tdl_a_profile(rms_delay_spread_ns=rms_delay_spread_ns)
    L_h = len(delays_s)

    # Random complex gains: alpha_l ~ CN(0, p_l)
    alpha = (np.sqrt(p_lin) / np.sqrt(2.0)) * (np.random.randn(L_h) + 1j * np.random.randn(L_h))

    # True CFR across all subcarriers: H[k] = sum_l alpha_l exp(-j2πkΔfτ_l)
    phase = np.exp(-1j * 2.0 * np.pi * np.outer(k_all * delta_f_hz, delays_s))
    H_true = phase @ alpha  # shape (N,)

    # Pilot symbols unit-modulus known
    X_p = np.ones(num_pilots, dtype=complex)
    H_p_true = H_true[k_p]
    signal_p = X_p * H_p_true

    # Recompute sigma from current signal power at EACH operating point
    signal_power = float(np.mean(np.abs(signal_p) ** 2))
    snr_linear = 10 ** (snr_db / 10.0)
    sigma2 = signal_power / (snr_linear + 1e-15)
    sigma = np.sqrt(sigma2)

    # Correct complex AWGN scaling (D1)
    noise = (sigma / np.sqrt(2.0)) * (np.random.randn(num_pilots) + 1j * np.random.randn(num_pilots))
    y_p = signal_p + noise

    return {
        "H_true": H_true,
        "k_p": k_p,
        "X_p": X_p,
        "y_p": y_p,
        "sigma2": float(sigma2),
        "delays_s": delays_s,
        "p_lin": p_lin
    }

Next block implements all required estimators in formula order.

**Inputs:** $\mathbf{y}_p,\mathbf{X}_p,k_p,\sigma_n^2,\{p_\ell,\tau_\ell\}$.  
**Outputs:** $\hat{\mathbf{H}}_{\mathrm{lin}},\hat{\mathbf{H}}_{\mathrm{spline}},\hat{\mathbf{H}}_{\mathrm{DFT}},\hat{\mathbf{H}}_{\mathrm{LMMSE}}$.  
Key equations:
- $\hat{\mathbf{h}}_p^{LS}=\mathbf{X}_p^{-1}\mathbf{y}_p$
- $\hat{\mathbf{H}}_{\mathrm{DFT}}=\mathcal{F}\{\mathcal{T}_{L_t}(\mathcal{F}^{-1}\hat{\mathbf{H}}_{\mathrm{lin}})\}$
- $\hat{\mathbf{H}}_{\mathrm{LMMSE}}=\mathbf{R}_{Hp}(\mathbf{R}_{pp}+\sigma_n^2\mathbf{I})^{-1}\hat{\mathbf{h}}_p^{LS}$

In [ ]:
def ls_pilot_estimate(y_p, X_p):
    # Formula: \hat{h}_p^{LS} = X_p^{-1} y_p (= y_p for unit pilots)
    return y_p / X_p


def periodic_linear_interpolation(k_p, h_p_ls, N):
    # Formula: H_lin[k] = I_lin{(k_p, h_p_ls)}(k)
    k_eval = np.arange(N)
    k_ext = np.concatenate([k_p, [N]])
    h_ext = np.concatenate([h_p_ls, [h_p_ls[0]]])  # periodic closure

    h_real = np.interp(k_eval, k_ext, np.real(h_ext))
    h_imag = np.interp(k_eval, k_ext, np.imag(h_ext))
    return h_real + 1j * h_imag


def periodic_spline_interpolation(k_p, h_p_ls, N):
    # Formula: H_spline[k] = sum_m h_p_ls[m] * beta_3((k-k_p,m)/Delta_k)
    # Implemented via periodic cubic spline interpolation.
    k_eval = np.arange(N)
    k_ext = np.concatenate([k_p, [N]])
    h_ext = np.concatenate([h_p_ls, [h_p_ls[0]]])

    cs_real = CubicSpline(k_ext, np.real(h_ext), bc_type='periodic')
    cs_imag = CubicSpline(k_ext, np.imag(h_ext), bc_type='periodic')
    return cs_real(k_eval) + 1j * cs_imag(k_eval)


def dft_truncated_ls_from_linear(H_lin, L_t):
    # Formula:
    # \tilde h[n] = (1/N) sum_k H_lin[k] e^{j2πkn/N}
    # h_T[n] = \tilde h[n] 1_{0<=n<L_t}
    # H_DFT[k] = sum_n h_T[n] e^{-j2πkn/N}
    h_ifft = np.fft.ifft(H_lin)
    h_trunc = np.zeros_like(h_ifft)
    h_trunc[:L_t] = h_ifft[:L_t]
    H_dft = np.fft.fft(h_trunc)
    return H_dft, h_ifft, h_trunc


def build_frequency_covariance(N, delta_f_hz, delays_s, p_lin):
    # Formula: [R_HH]_{m,n} = sum_l p_l exp(-j2π(m-n)Δf τ_l)
    idx = np.arange(N)
    lag = idx[:, None] - idx[None, :]
    R_HH = np.zeros((N, N), dtype=complex)
    for l in range(len(delays_s)):
        R_HH += p_lin[l] * np.exp(-1j * 2.0 * np.pi * lag * delta_f_hz * delays_s[l])
    return R_HH


def lmmse_wiener_interpolation(h_p_ls, k_p, sigma2, N, delta_f_hz, delays_s, p_lin):
    # Formula: H_LMMSE = R_Hp (R_pp + sigma2 I)^(-1) h_p_ls
    R_HH = build_frequency_covariance(N, delta_f_hz, delays_s, p_lin)
    R_Hp = R_HH[:, k_p]
    R_pp = R_HH[np.ix_(k_p, k_p)]
    reg = sigma2 * np.eye(len(k_p), dtype=complex)

    # explicit inversion equivalent via solve; dimension <= 16x16 (or chosen pilot count)
    w = np.linalg.solve(R_pp + reg, h_p_ls)
    H_lmmse = R_Hp @ w
    return H_lmmse


def nmse_nonpilot(H_hat, H_true, k_p, N):
    data_idx = np.setdiff1d(np.arange(N), k_p)
    num = np.sum(np.abs(H_hat[data_idx] - H_true[data_idx]) ** 2)
    den = np.sum(np.abs(H_true[data_idx]) ** 2) + 1e-15
    return float(num / den)


def estimate_flops_per_symbol(N, Np):
    # Rough operation counts (for comparison only)
    flops = {
        "LS_linear_interpolation": float(8 * N),                  # interpolation + light ops
        "LS_spline_interpolation": float(25 * N),                 # higher constant than linear
        "DFT_truncated_LS": float(10 * N * np.log2(N)),           # FFT + IFFT
        "LMMSE_Wiener_frequency_interpolation": float((2/3) * (Np ** 3) + 8 * N * (Np ** 2))
    }
    return flops

Now we build the full Monte Carlo evaluation sweep.

**What it computes:** NMSE-vs-SNR for all methods, 95% CI bands, runtime/ms, FLOPs, and secondary-factor sensitivity.

**Inputs:** `eval_config` with SNR points, trials, OFDM/channel hyperparameters.  
**Outputs:** JSON-serializable `performance_data` dict.

Important loop structure:
- For each SNR point, regenerate data each trial.
- Compute every method independently per trial and per SNR.
- Aggregate $\overline{\mathrm{NMSE}}$, CI, runtime.
- Add curve sanity guards and constant-result warnings.

In [ ]:
def run_evaluation(eval_config):
    # Named hyperparameters (D10)
    N_SC = int(eval_config["num_subcarriers"])
    DELTA_F = float(eval_config["subcarrier_spacing_hz"])
    NUM_MC = int(eval_config["num_monte_carlo"])
    SNR_POINTS = list(eval_config["variable_points"])
    NUM_PILOTS = int(eval_config["num_pilot_subcarriers"])
    RMS_DELAY_NS = float(eval_config["rms_delay_spread_ns"])
    L_T_DEFAULT = int(eval_config["dft_truncation_length"])

    METHOD_NAMES = [
        "LS_linear_interpolation",
        "LS_spline_interpolation",
        "DFT_truncated_LS",
        "LMMSE_Wiener_frequency_interpolation"
    ]

    nmse_trials_db = {m: [] for m in METHOD_NAMES}
    mean_nmse_db = {m: [] for m in METHOD_NAMES}
    ci_low_db = {m: [] for m in METHOD_NAMES}
    ci_high_db = {m: [] for m in METHOD_NAMES}
    runtime_ms = {m: [] for m in METHOD_NAMES}

    # Sensitivity curves required: DFT truncation length impact vs SNR
    LT_LIST = [4, 8, 12, 16]
    dft_lt_curves_db = {f"DFT_Lt{lt}": [] for lt in LT_LIST}

    delay_example = None

    for snr_db in SNR_POINTS:
        nmse_lin_per_method = {m: [] for m in METHOD_NAMES}
        time_acc = {m: 0.0 for m in METHOD_NAMES}

        nmse_lin_dft_by_lt = {lt: [] for lt in LT_LIST}

        for _ in range(NUM_MC):
            obs = generate_ofdm_observation(
                snr_db=snr_db,
                N=N_SC,
                delta_f_hz=DELTA_F,
                num_pilots=NUM_PILOTS,
                rms_delay_spread_ns=RMS_DELAY_NS
            )

            H_true = obs["H_true"]
            k_p = obs["k_p"]
            X_p = obs["X_p"]
            y_p = obs["y_p"]
            sigma2 = obs["sigma2"]
            delays_s = obs["delays_s"]
            p_lin = obs["p_lin"]

            # Step 2: LS pilot estimate
            h_p_ls = ls_pilot_estimate(y_p, X_p)

            # LS linear interpolation
            t0 = time.perf_counter()
            H_lin = periodic_linear_interpolation(k_p, h_p_ls, N_SC)
            time_acc["LS_linear_interpolation"] += (time.perf_counter() - t0)
            nmse_lin_per_method["LS_linear_interpolation"].append(nmse_nonpilot(H_lin, H_true, k_p, N_SC))

            # LS spline interpolation
            t0 = time.perf_counter()
            H_spline = periodic_spline_interpolation(k_p, h_p_ls, N_SC)
            time_acc["LS_spline_interpolation"] += (time.perf_counter() - t0)
            nmse_lin_per_method["LS_spline_interpolation"].append(nmse_nonpilot(H_spline, H_true, k_p, N_SC))

            # DFT truncation default
            t0 = time.perf_counter()
            H_dft, h_ifft, h_trunc = dft_truncated_ls_from_linear(H_lin, L_T_DEFAULT)
            time_acc["DFT_truncated_LS"] += (time.perf_counter() - t0)
            nmse_lin_per_method["DFT_truncated_LS"].append(nmse_nonpilot(H_dft, H_true, k_p, N_SC))

            # LMMSE Wiener interpolation
            t0 = time.perf_counter()
            H_lmmse = lmmse_wiener_interpolation(h_p_ls, k_p, sigma2, N_SC, DELTA_F, delays_s, p_lin)
            time_acc["LMMSE_Wiener_frequency_interpolation"] += (time.perf_counter() - t0)
            nmse_lin_per_method["LMMSE_Wiener_frequency_interpolation"].append(nmse_nonpilot(H_lmmse, H_true, k_p, N_SC))

            # DFT Lt sensitivity curves
            for lt in LT_LIST:
                H_dft_lt, _, _ = dft_truncated_ls_from_linear(H_lin, lt)
                nmse_lin_dft_by_lt[lt].append(nmse_nonpilot(H_dft_lt, H_true, k_p, N_SC))

            # Save one delay-domain visualization sample
            if delay_example is None and snr_db == SNR_POINTS[len(SNR_POINTS)//2]:
                h_true_ifft = np.fft.ifft(H_true)
                delay_example = {
                    "n": list(np.arange(N_SC)),
                    "true_delay_taps": list(np.abs(h_true_ifft)),
                    "ifft_of_ls_linear": list(np.abs(h_ifft)),
                    "after_truncation": list(np.abs(h_trunc))
                }

        # Aggregate per SNR
        for m in METHOD_NAMES:
            vals_lin = np.array(nmse_lin_per_method[m], dtype=float)
            vals_db = 10.0 * np.log10(vals_lin + 1e-15)
            nmse_trials_db[m].append(vals_db.tolist())

            mean_db = float(np.mean(vals_db))
            std_db = float(np.std(vals_db, ddof=1))
            ci = 1.96 * std_db / np.sqrt(NUM_MC)

            mean_nmse_db[m].append(mean_db)
            ci_low_db[m].append(mean_db - ci)
            ci_high_db[m].append(mean_db + ci)
            runtime_ms[m].append(float((time_acc[m] / NUM_MC) * 1e3))

        for lt in LT_LIST:
            vals_lt_db = 10.0 * np.log10(np.array(nmse_lin_dft_by_lt[lt]) + 1e-15)
            dft_lt_curves_db[f"DFT_Lt{lt}"].append(float(np.mean(vals_lt_db)))

    # Secondary factor (requirement 19): "snapshot_sensitivity" key populated
    # Here we use pilot density sensitivity at fixed SNR.
    FIXED_SNR_DB = 15
    PILOT_COUNTS = [8, 16, 32]
    sec_mc = min(200, NUM_MC)
    sec_proposed = []
    sec_baseline = []
    for npil in PILOT_COUNTS:
        vals_prop = []
        vals_base = []
        for _ in range(sec_mc):
            obs = generate_ofdm_observation(
                snr_db=FIXED_SNR_DB,
                N=N_SC,
                delta_f_hz=DELTA_F,
                num_pilots=npil,
                rms_delay_spread_ns=RMS_DELAY_NS
            )
            H_true = obs["H_true"]
            k_p = obs["k_p"]
            h_p_ls = ls_pilot_estimate(obs["y_p"], obs["X_p"])

            H_lin = periodic_linear_interpolation(k_p, h_p_ls, N_SC)
            H_lmmse = lmmse_wiener_interpolation(
                h_p_ls, k_p, obs["sigma2"], N_SC, DELTA_F, obs["delays_s"], obs["p_lin"]
            )
            vals_base.append(nmse_nonpilot(H_lin, H_true, k_p, N_SC))
            vals_prop.append(nmse_nonpilot(H_lmmse, H_true, k_p, N_SC))
        sec_baseline.append(float(10*np.log10(np.mean(vals_base) + 1e-15)))
        sec_proposed.append(float(10*np.log10(np.mean(vals_prop) + 1e-15)))

    # D9 sanity checks
    for method_name in METHOD_NAMES:
        arr = mean_nmse_db[method_name]
        assert len(arr) == len(SNR_POINTS), f"{method_name}: result length mismatch"
        if len(set(round(v, 8) for v in arr)) == 1:
            print(f"WARNING: {method_name} produced constant results across all operating points — likely implementation bug.")
        assert np.all(np.isfinite(arr)), f"{method_name}: non-finite values detected"

    for k, arr in dft_lt_curves_db.items():
        assert len(arr) == len(SNR_POINTS), f"{k}: result length mismatch"
        if len(set(round(v, 8) for v in arr)) == 1:
            print(f"WARNING: {k} produced constant results across all operating points — likely implementation bug.")

    flops = estimate_flops_per_symbol(N_SC, NUM_PILOTS)

    performance_data = {
        "metric_curve": {
            "x": [float(x) for x in SNR_POINTS],
            "LS_linear_interpolation": [float(v) for v in mean_nmse_db["LS_linear_interpolation"]],
            "LS_spline_interpolation": [float(v) for v in mean_nmse_db["LS_spline_interpolation"]],
            "DFT_truncated_LS": [float(v) for v in mean_nmse_db["DFT_truncated_LS"]],
            "LMMSE_Wiener_frequency_interpolation": [float(v) for v in mean_nmse_db["LMMSE_Wiener_frequency_interpolation"]],
            "x_label": "SNR (dB)",
            "y_label": "NMSE on non-pilot tones (dB)"
        },
        "metric_ci95": {
            "LS_linear_interpolation_low": [float(v) for v in ci_low_db["LS_linear_interpolation"]],
            "LS_linear_interpolation_high": [float(v) for v in ci_high_db["LS_linear_interpolation"]],
            "LS_spline_interpolation_low": [float(v) for v in ci_low_db["LS_spline_interpolation"]],
            "LS_spline_interpolation_high": [float(v) for v in ci_high_db["LS_spline_interpolation"]],
            "DFT_truncated_LS_low": [float(v) for v in ci_low_db["DFT_truncated_LS"]],
            "DFT_truncated_LS_high": [float(v) for v in ci_high_db["DFT_truncated_LS"]],
            "LMMSE_Wiener_frequency_interpolation_low": [float(v) for v in ci_low_db["LMMSE_Wiener_frequency_interpolation"]],
            "LMMSE_Wiener_frequency_interpolation_high": [float(v) for v in ci_high_db["LMMSE_Wiener_frequency_interpolation"]]
        },
        "runtime_ms_per_symbol": {
            m: float(np.mean(runtime_ms[m])) for m in METHOD_NAMES
        },
        "complexity_flops_per_symbol": {k: float(v) for k, v in flops.items()},
        "nmse_gain_over_ls_linear_dB": {
            "DFT_truncated_LS": [float(a - b) for a, b in zip(mean_nmse_db["LS_linear_interpolation"], mean_nmse_db["DFT_truncated_LS"])],
            "LMMSE_Wiener_frequency_interpolation": [float(a - b) for a, b in zip(mean_nmse_db["LS_linear_interpolation"], mean_nmse_db["LMMSE_Wiener_frequency_interpolation"])]
        },
        "dft_truncation_sensitivity_curves": {
            "x": [float(x) for x in SNR_POINTS],
            "DFT_Lt4": [float(v) for v in dft_lt_curves_db["DFT_Lt4"]],
            "DFT_Lt8": [float(v) for v in dft_lt_curves_db["DFT_Lt8"]],
            "DFT_Lt12": [float(v) for v in dft_lt_curves_db["DFT_Lt12"]],
            "DFT_Lt16": [float(v) for v in dft_lt_curves_db["DFT_Lt16"]],
            "x_label": "SNR (dB)",
            "y_label": "NMSE (dB)"
        },
        "snapshot_sensitivity": {
            "x": [int(v) for v in PILOT_COUNTS],
            "proposed": [float(v) for v in sec_proposed],
            "baseline_name": [float(v) for v in sec_baseline],
            "x_label": "Number of pilots (fixed SNR=15 dB)",
            "y_label": "NMSE (dB)"
        },
        "delay_domain_example": delay_example
    }

    return performance_data

# AUTO-WISP RESULTS HELPER
def _autowisp_normalize_results(candidate):
    if candidate is None:
        candidate = {}
    if not isinstance(candidate, dict):
        candidate = {'performance_data': {'raw_results': candidate}}
    performance_data = candidate.get('performance_data')
    if not isinstance(performance_data, dict):
        for key in ('metrics', 'results', 'evaluation_results', 'raw_results'):
            value = candidate.get(key)
            if isinstance(value, dict):
                performance_data = value
                break
    if not isinstance(performance_data, dict):
        performance_data = {'raw_results': performance_data if performance_data is not None else {}}
    report_assets = candidate.get('report_assets')
    if not isinstance(report_assets, dict):
        report_assets = {}
    report_assets.setdefault('problem_summary', 'Recovered from notebook auto-debug path.')
    report_assets.setdefault('solution_summary', 'Execution contract was normalized to ensure RESULTS generation.')
    report_assets.setdefault('evaluation_summary', 'Primary metric exported as NMSE_dB_nonpilot.')
    performance_data = _autowisp_to_builtin(performance_data)
    tables = report_assets.get('tables')
    if not isinstance(tables, list) or not tables:
        tables = _autowisp_build_tables(performance_data)
    report_assets['tables'] = tables
    figures = report_assets.get('figures') or report_assets.get('figures_metadata') or []
    if not isinstance(figures, list):
        figures = []
    if not figures:
        figures = _autowisp_build_figures(performance_data)
    report_assets['figures'] = figures
    report_assets['figures_metadata'] = figures
    candidate['algorithm'] = candidate.get('algorithm') or 'Notebook Auto-Debug'
    candidate['elapsed_sec'] = float(candidate.get('elapsed_sec') or 0.0)
    candidate['performance_data'] = performance_data
    candidate['report_assets'] = report_assets
    return candidate

def _autowisp_to_builtin(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(k): _autowisp_to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [_autowisp_to_builtin(v) for v in value]
    if hasattr(value, 'tolist'):
        return _autowisp_to_builtin(value.tolist())
    if hasattr(value, 'item'):
        try:
            return value.item()
        except Exception:
            pass
    return str(value)

def _autowisp_curve_specs(perf_data):
    if not isinstance(perf_data, dict):
        return []
    _x_keys = ('x', 'variable_points', 'operating_points', 'snr', 'snr_points', 'snr_db')
    top_x = None
    for key in _x_keys:
        value = perf_data.get(key)
        if isinstance(value, list) and len(value) >= 2:
            top_x = value
            break
    specs = []
    for metric_name, payload in perf_data.items():
        if not isinstance(payload, dict):
            continue
        x_values = None
        for key in _x_keys:
            value = payload.get(key)
            if isinstance(value, list) and len(value) >= 2:
                x_values = value
                break
        if x_values is None:
            x_values = top_x
        if not isinstance(x_values, list) or len(x_values) < 2:
            continue
        _skip_keys = set(_x_keys) | {'x_label', 'xlabel', 'ylabel', 'title'}
        series = {}
        for series_name, series_values in payload.items():
            if series_name in _skip_keys:
                continue
            if isinstance(series_values, list) and len(series_values) == len(x_values):
                series[str(series_name)] = series_values
        if series:
            specs.append({
                'metric': str(metric_name),
                'x': x_values,
                'x_label': payload.get('x_label') or payload.get('xlabel') or 'Operating Point',
                'series': series,
            })
    return specs

def _autowisp_build_tables(perf_data):
    tables = []
    for spec in _autowisp_curve_specs(perf_data):
        columns = [spec['x_label']] + [name.replace('_', ' ').title() for name in spec['series'].keys()]
        rows = []
        for idx, x_value in enumerate(spec['x']):
            rows.append([x_value] + [values[idx] for values in spec['series'].values()])
        tables.append({
            'title': spec['metric'].replace('_', ' ').title(),
            'columns': columns,
            'rows': rows,
        })
    return tables

def _autowisp_build_figures(perf_data):
    figures = []
    for spec in _autowisp_curve_specs(perf_data):
        figures.append({
            'title': spec['metric'].replace('_', ' ').title(),
            'xlabel': spec['x_label'],
            'ylabel': spec['metric'].replace('_', ' '),
            'x': spec['x'],
            'series': spec['series'],
        })
    return figures

if 'run_experiment' not in globals():
    def run_experiment(eval_config=None):
        eval_config = dict(eval_config or globals().get('EVAL_CONFIG') or {})
        existing_runner = globals().get('run_evaluation')
        if callable(existing_runner):
            return _autowisp_normalize_results(existing_runner(eval_config))
        existing_results = globals().get('RESULTS')
        if isinstance(existing_results, dict):
            return _autowisp_normalize_results(existing_results)
        for key in ('performance_data', 'metrics', 'results', 'evaluation_results'):
            value = globals().get(key)
            if isinstance(value, dict):
                return _autowisp_normalize_results({'performance_data': value})
        raise RuntimeError('Notebook auto-debug could not infer performance_data for RESULTS generation')



The next block executes the experiment end-to-end with reproducibility controls.

**What it computes:** sets seed, defines all hyperparameters, runs evaluation, measures wall-clock time, and builds `RESULTS`.

**Required output object:**  
`RESULTS = {algorithm, elapsed_sec, performance_data, report_assets}`

In [ ]:
np.random.seed(42)

# Named hyperparameters
NUM_SUBCARRIERS = 64
SUBCARRIER_SPACING_HZ = 15e3
CP_LENGTH = 16
NUM_PILOTS = 16
RMS_DELAY_SPREAD_NS = 30.0
SNR_POINTS_DB = list(range(0, 31, 2))
NUM_MONTE_CARLO = 500
DFT_TRUNC_LEN = 8

eval_config = {
    "num_subcarriers": NUM_SUBCARRIERS,
    "subcarrier_spacing_hz": SUBCARRIER_SPACING_HZ,
    "cp_length_samples": CP_LENGTH,
    "num_pilot_subcarriers": NUM_PILOTS,
    "rms_delay_spread_ns": RMS_DELAY_SPREAD_NS,
    "num_monte_carlo": NUM_MONTE_CARLO,
    "variable_points": SNR_POINTS_DB,
    "dft_truncation_length": DFT_TRUNC_LEN
}

tic = time.perf_counter()
performance_data = run_evaluation(eval_config)
elapsed = time.perf_counter() - tic

table_asset = {
    "name": "method_summary",
    "columns": ["method", "avg_nmse_dB_over_snr", "avg_runtime_ms", "flops_per_symbol"],
    "rows": []
}
for m in [
    "LS_linear_interpolation",
    "LS_spline_interpolation",
    "DFT_truncated_LS",
    "LMMSE_Wiener_frequency_interpolation"
]:
    avg_nmse = float(np.mean(performance_data["metric_curve"][m]))
    avg_rt = float(performance_data["runtime_ms_per_symbol"][m])
    fl = float(performance_data["complexity_flops_per_symbol"][m])
    table_asset["rows"].append([m, avg_nmse, avg_rt, fl])

RESULTS = {
    "algorithm": "Comparative OFDM CFR estimation: LS-linear/spline, DFT-truncated LS, LMMSE-Wiener",
    "elapsed_sec": float(elapsed),
    "performance_data": performance_data,
    "report_assets": {
        "problem_summary": "Estimate full 64-tone CFR from 16 noisy comb pilots and evaluate non-pilot NMSE vs SNR.",
        "solution_summary": "Implemented all required baselines plus covariance-aided LMMSE; included DFT-Lt and pilot-density sensitivity.",
        "evaluation_summary": "500 MC trials/SNR over 0:2:30 dB with 95% CI bands, runtime and FLOPs comparison.",
        "tables": [table_asset],
        "figures": []
    }
}

# AUTO-WISP EXECUTION CONTRACT
if not isinstance(globals().get('EVAL_CONFIG'), dict):
    EVAL_CONFIG = {}
if not isinstance(globals().get('RESULTS'), dict):
    RESULTS = run_experiment(globals().get('EVAL_CONFIG', {}))
RESULTS = _autowisp_normalize_results(RESULTS)
RESULTS.setdefault('notes', {})['primary_metric'] = 'NMSE_dB_nonpilot'



Next block generates and saves the three required figures.

**Figure 1:** NMSE vs SNR for all methods with CI bands.  
**Figure 2:** DFT truncation-length sensitivity ($L_t\in\{4,8,12,16\}$) vs SNR.  
**Figure 3:** delay-domain denoising visualization using $|\tilde h[n]|$ before/after truncation.

For lower-is-better NMSE, we use semilogy on linear-scale NMSE values:
$$
\mathrm{NMSE}_{lin}=10^{\mathrm{NMSE}_{dB}/10}.
$$

In [ ]:
perf = RESULTS["performance_data"]
snr = np.array(perf["metric_curve"]["x"])

# Fig 1: Primary comparison
fig1, ax1 = plt.subplots(figsize=(7, 4.5))
methods = [
    "LS_linear_interpolation",
    "LS_spline_interpolation",
    "DFT_truncated_LS",
    "LMMSE_Wiener_frequency_interpolation"
]
for m in methods:
    y_db = np.array(perf["metric_curve"][m])
    y_lin = 10 ** (y_db / 10.0)
    ax1.semilogy(snr, y_lin, marker='o', linewidth=1.8, label=m)

    lo_db = np.array(perf["metric_ci95"][f"{m}_low"])
    hi_db = np.array(perf["metric_ci95"][f"{m}_high"])
    ax1.fill_between(snr, 10 ** (lo_db / 10.0), 10 ** (hi_db / 10.0), alpha=0.12)

ax1.set_xlabel("SNR (dB)")
ax1.set_ylabel("NMSE (linear, non-pilot)")
ax1.set_title("Fig 1: NMSE vs SNR (all methods)")
ax1.grid(True, which='both', ls='--', alpha=0.4)
ax1.legend(fontsize=8)
fig1.savefig("fig1_nmse_vs_snr.png", dpi=150, bbox_inches='tight')

# Fig 2: DFT truncation sensitivity
fig2, ax2 = plt.subplots(figsize=(7, 4.5))
for key in ["DFT_Lt4", "DFT_Lt8", "DFT_Lt12", "DFT_Lt16"]:
    y_db = np.array(perf["dft_truncation_sensitivity_curves"][key])
    ax2.semilogy(snr, 10 ** (y_db / 10.0), marker='s', linewidth=1.8, label=key)

ax2.set_xlabel("SNR (dB)")
ax2.set_ylabel("NMSE (linear)")
ax2.set_title("Fig 2: Sensitivity to DFT truncation length")
ax2.grid(True, which='both', ls='--', alpha=0.4)
ax2.legend()
fig2.savefig("fig2_dft_truncation_sensitivity.png", dpi=150, bbox_inches='tight')

# Fig 3: Delay-domain denoising visualization
ex = perf["delay_domain_example"]
n = np.array(ex["n"])
fig3, ax3 = plt.subplots(figsize=(7, 4.5))
ax3.plot(n, ex["true_delay_taps"], linewidth=2, label="true_delay_taps")
ax3.plot(n, ex["ifft_of_ls_linear"], linewidth=1.6, label="ifft_of_ls_linear")
ax3.plot(n, ex["after_truncation"], linewidth=1.6, label="after_truncation")
ax3.set_xlabel("Delay tap index n")
ax3.set_ylabel("|h[n]|")
ax3.set_title("Fig 3: Delay-domain denoising behavior")
ax3.grid(True, ls='--', alpha=0.4)
ax3.legend()
fig3.savefig("fig3_delay_domain_denoising.png", dpi=150, bbox_inches='tight')

RESULTS["report_assets"]["figures"] = [
    "fig1_nmse_vs_snr.png",
    "fig2_dft_truncation_sensitivity.png",
    "fig3_delay_domain_denoising.png"
]

### Result Notes

| Method | Complexity Trend | Expected NMSE Rank (best→worst) | Runtime Trend |
|---|---:|---:|---:|
| LS_linear_interpolation | Lowest | 4 | Fastest |
| LS_spline_interpolation | Low | 3/4 (scenario-dependent) | Fast |
| DFT_truncated_LS | Moderate (FFT-based) | 2/3 | Moderate |
| LMMSE_Wiener_frequency_interpolation | Highest (pilot-domain solve) | 1 | Slowest |

### Key-point comparison table (non-pilot NMSE, dB)

| SNR (dB) | LS-linear | LS-spline | DFT-truncated | LMMSE-Wiener |
|---:|---:|---:|---:|---:|
| 0 | from `RESULTS["performance_data"]["metric_curve"]` | from results | from results | from results |
| 10 | from results | from results | from results | from results |
| 20 | from results | from results | from results | from results |
| 30 | from results | from results | from results | from results |

### Interpretation
- As SNR increases, NMSE decreases for all methods (sanity check for noise scaling).
- DFT truncation shows a bias-variance trade-off through $L_t$.
- LMMSE Wiener uses PDP-based covariance and is expected to provide best NMSE at the cost of higher compute.

# estimation|recovery|interpolation

This notebook implements the **LS-Interpolation + DFT-Denoising + Pilot-Domain LMMSE Wiener** solution as a single executable workflow.